# 9 · PCA and K-Means on Micrographs
Runs a sliding-window patch extraction over each micrograph, encodes patches with the frozen DINOv2 backbone, then saves per-micrograph PCA visualisations and K-Means segmentation masks.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
DEVICE           = 'cuda:0'
BASE_PATH        = './metadata/DINOv2_checkpoint/'
MICROGRAPH_DIR   = './metadata/micrographs/'

BATCH_SIZE       = 1024   # patches per forward pass
IM_SIZE          = 224    # patch side length (px)
STRIDE           = 16     # sliding-window stride (px)
N_LAST_BLOCKS    = 1      # must match the value used during linear eval
PCA_COMPONENTS   = 3
KMEANS_CENTERS   = 2

# Dataset normalisation constants (computed over the training set)
NORMALIZE_MEAN   = 0.5475857690770103
NORMALIZE_STD    = 0.12047245612891117

In [2]:
import os
from functools import partial

import mrcfile
import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from dinov3.eval.linear import create_linear_input
from dinov3.eval.setup import ModelConfig, load_model_and_context
from dinov3.eval.utils import ModelWithIntermediateLayers
from utils.utils import MRC, normalize_single_image_array

## 1 · Initialise distributed context and load backbone

In [3]:
if not dist.is_initialized():
    dist.init_process_group(
        backend='nccl',
        rank=0,
        world_size=1,
        init_method='tcp://127.0.0.1:29400',
    )

torch.cuda.set_device(int(DEVICE.split(':')[-1]))

model_cfg = ModelConfig(
    config_file=f'{BASE_PATH}config.yaml',
    pretrained_weights=f'{BASE_PATH}teacher_checkpoint.pth',
    dino_hub=None,
)
backbone, ctx = load_model_and_context(model_cfg, output_dir=BASE_PATH)
backbone.to(DEVICE).eval()

# n_last_blocks must match what was used during linear eval
autocast_ctx  = partial(torch.autocast, device_type='cuda', enabled=True, dtype=ctx['autocast_dtype'])
feature_model = ModelWithIntermediateLayers(backbone, N_LAST_BLOCKS, autocast_ctx)

print('Backbone ready on', DEVICE)

Backbone ready on cuda:0


## 2 · Process micrographs

In [4]:
mrc_files = sorted(f for f in os.listdir(MICROGRAPH_DIR) if f.endswith('.mrc'))
print(f'Found {len(mrc_files)} micrograph(s)')

for di, filename in enumerate(mrc_files):
    emp_id   = filename.split('_')[0]
    mrc_path = os.path.join(MICROGRAPH_DIR, filename)

    # ── Load ────────────────────────────────────────────────────────────────
    try:
        with mrcfile.open(mrc_path, 'r') as f:
            m = f.data.astype(float)
    except Exception:  # fallback for non-standard MRC files
        m = MRC(mrc_path).data[:, :, 0].astype(float)

    m_shape = m.shape
    H, W    = m_shape

    # ── Save normalized PNG ──────────────────────────────────────────────────
    m_norm = normalize_single_image_array(m)
    Image.fromarray((m_norm * 255).astype(np.uint8), mode='L').save(
        os.path.join(MICROGRAPH_DIR, f'{filename}.png')
    )

    # ── Sliding-window patch grid ────────────────────────────────────────────
    rss_x    = (H - IM_SIZE) // STRIDE + 1
    rss_y    = (W - IM_SIZE) // STRIDE + 1
    grid_h   = (rss_x - 1) * STRIDE + IM_SIZE
    grid_w   = (rss_y - 1) * STRIDE + IM_SIZE
    offset_x = (H - grid_h) // 2
    offset_y = (W - grid_w) // 2

    im_tensor = []
    for ix in range(rss_x):
        x0 = offset_x + ix * STRIDE
        for iy in range(rss_y):
            y0 = offset_y + iy * STRIDE
            im_tensor.append(m[x0:x0 + IM_SIZE, y0:y0 + IM_SIZE])
    im_tensor = torch.from_numpy(np.array(im_tensor)).float()

    # ── Normalise patches ────────────────────────────────────────────────────
    patches     = im_tensor.unsqueeze(1)  # (N, 1, H, W)
    patches_min = patches.amin(dim=(-1, -2), keepdim=True)
    patches_max = patches.amax(dim=(-1, -2), keepdim=True)
    patches     = (patches - patches_min) / (patches_max - patches_min + 1e-8)
    patches     = 1 - patches                        # invert contrast
    patches     = (patches - NORMALIZE_MEAN) / NORMALIZE_STD

    # ── Extract DINOv2 features ──────────────────────────────────────────────
    N       = patches.shape[0]
    latents = []
    with torch.inference_mode():
        with torch.autocast(device_type='cuda', dtype=ctx['autocast_dtype']):
            for start in range(0, N, BATCH_SIZE):
                batch = patches[start:start + BATCH_SIZE].to(DEVICE)
                feats = create_linear_input(
                    feature_model(batch.float()), N_LAST_BLOCKS, True
                ).cpu().float().numpy()
                latents.append(feats)
                print(f'  [{di+1}/{len(mrc_files)}] {filename}  {start+len(feats)}/{N}', end='\r')
    latents = np.concatenate(latents)  # (N, D)
    print()

    # ── Shared upsampling helper ─────────────────────────────────────────────
    def to_fullres(grid_tensor):
        """Pad and upsample a (1, C, rss_x, rss_y) grid to the original micrograph size."""
        tp = ((H // STRIDE) - rss_x) // 2
        lr = ((W // STRIDE) - rss_y) // 2
        if tp or lr:
            grid_tensor = F.pad(grid_tensor, (lr, lr, tp, tp), mode='constant', value=0)
        return F.interpolate(grid_tensor, size=m_shape, mode='nearest')[0]

    # ── PCA ──────────────────────────────────────────────────────────────────
    pca       = PCA(n_components=PCA_COMPONENTS, whiten=True).fit(latents)
    projected = torch.from_numpy(pca.transform(latents))  # (N, PCA_COMPONENTS)

    proj_tensor = projected.reshape(rss_x, rss_y, PCA_COMPONENTS).permute(2, 0, 1).unsqueeze(0).float()
    proj_up     = to_fullres(proj_tensor).permute(1, 2, 0).numpy()  # (H, W, C)

    proj_min  = proj_up.min(axis=(0, 1), keepdims=True)
    proj_max  = proj_up.max(axis=(0, 1), keepdims=True)
    proj_u8   = ((proj_up - proj_min) / (proj_max - proj_min + 1e-8) * 255).astype(np.uint8)

    Image.fromarray(proj_u8[:, :, 0], mode='L').save(
        os.path.join(MICROGRAPH_DIR, f'{emp_id}_PCA_1_mask_{IM_SIZE}_{STRIDE}.png')
    )
    Image.fromarray(proj_u8, mode='RGB').save(
        os.path.join(MICROGRAPH_DIR, f'{emp_id}_PCA_{PCA_COMPONENTS}_mask_{IM_SIZE}_{STRIDE}.png')
    )

    # ── K-Means ───────────────────────────────────────────────────────────────
    labels = KMeans(n_clusters=KMEANS_CENTERS, n_init=10).fit_predict(latents)

    km_tensor = (
        torch.from_numpy(labels.reshape(rss_x, rss_y).astype(np.float32))
        .unsqueeze(0).unsqueeze(0)
    )
    km_up  = to_fullres(km_tensor)[0].numpy()  # (H, W)
    km_min, km_max = km_up.min(), km_up.max()
    km_u8  = ((km_up - km_min) / (km_max - km_min + 1e-8) * 255).astype(np.uint8)

    Image.fromarray(km_u8, mode='L').save(
        os.path.join(MICROGRAPH_DIR, f'{emp_id}_kmeans_{KMEANS_CENTERS}_centers_{IM_SIZE}_{STRIDE}.png')
    )
print('Done')

Found 7 micrograph(s)
  [1/7] 10005_stack_0530_2x_SumCorr.mrc  47524/47524
Done
